# DSAR × Spark Declarative Pipelines · 01 · The pipeline (hardened)

**This notebook is the pipeline source.** Do **not** "Run all" it interactively —
attach it as the source of a **Lakeflow Declarative Pipeline** (Jobs & Pipelines →
Create pipeline → add this notebook). Databricks executes it as the pipeline
graph.

It mirrors Allegiant's Merlot topology and hardens it so a DSAR erasure on any
layer can **never** trigger the `append-only source` fatal failure:

```
raw_user ──stream──▶ bronze_user ──stream/AUTO CDC──▶ silver_user     gold_user
           (obfuscate PII inline)     (SCD type 1)              (materialized view)
                                                          bronze ──batch recompute──▶ gold
```

### The one line that fixes the streaming hops

Every **streaming** read that consumes a table an erasure will mutate uses:

```python
spark.readStream.option("skipChangeCommits", "true").table(...)
```

`skipChangeCommits` tells the stream: *when you meet a commit that contains an
UPDATE/DELETE (a non-append commit), skip that commit instead of failing.* It
does **not** undo the delete — the row is still gone from the source; the stream
just doesn't crash when it notices the source changed.

**Answering "we'd need it on both layers, right?":** yes — on the **two
append-only streaming hops**, `raw → bronze` and `bronze → silver`. The erasure
notebook (`02`) deletes at every layer, so each of those streaming reads meets a
non-append commit and needs the option (set on both here).

The aggregating **gold** layer is deliberately a **materialized view (batch
recompute), not a stream** — see section 3 for why (a `skipChangeCommits` stream
there would drop legitimate SCD1 updates, and a streaming aggregation could
resurrect an erased subject from checkpoint state). So: **`skipChangeCommits` on
the two streams; materialized view for gold.**

> Latency: skipping a commit just advances the stream's offset past that commit —
> it reads no data and scans no files, so the overhead is negligible (Dipankar's
> "it should not add too much latency").

## 0. Pipeline parameters

In a Declarative Pipeline these come from the **pipeline configuration**
(Settings → Advanced → Configuration), not notebook widgets. We read them with
`spark.conf.get(...)` and fall back to the `00` defaults so the notebook is also
readable standalone.

Set in the pipeline config:

| Key | Value |
|-----|-------|
| `dsar.catalog` | `dkushari_uc` |
| `dsar.schema`  | `allegiant_air_sdp_dsar` |

Also set the pipeline's **default catalog & target schema** to the same values so
the published tables land next to `raw_user`.

In [ ]:
import dlt
from pyspark.sql import functions as F

def cfg(key, default):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

CATALOG = cfg("dsar.catalog", "dkushari_uc")
SCHEMA  = cfg("dsar.schema",  "allegiant_air_sdp_dsar")
FQ      = f"{CATALOG}.{SCHEMA}"
SOURCE  = f"{FQ}.raw_user"   # the landing table built by notebook 00

print("Pipeline reads landing source:", SOURCE)

## 1. Bronze — obfuscate PII inline, streaming, append-only target

This single streaming table folds Allegiant's *"view (obfuscation) → bronze
streaming"* into one hop: it reads the raw landing table as a stream and applies
the **native-SQL** PII masking (the same `regexp_replace` approach as the masking
repo — scalar columns + in-JSON values) as it writes.

- `user_id`, `revenue`, `event_ts` are **preserved** (non-PII).
- `email`, `full_name`, and the `contact.*` values inside `profile_json` are
  **REDACTED**.
- **`skipChangeCommits` on the read of `raw_user`**: if an erasure later runs a
  `DELETE`/`UPDATE` on `raw_user`, this bronze stream skips that non-append commit
  instead of failing.

In [ ]:
REDACT = "***REDACTED***"

def _mask_json(col):
    # in-JSON masking, native SQL — quote-anchored keys so "name" != "appName"
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    return e

@dlt.table(
    name="bronze_user",
    comment="Streaming bronze: PII obfuscated inline; user_id/revenue preserved. Append-only target.",
    table_properties={"quality": "bronze"},
)
def bronze_user():
    src = (
        spark.readStream
        .option("skipChangeCommits", "true")   # ← survive erasure on raw_user
        .table(SOURCE)
    )
    return src.select(
        "event_id",
        "user_id",                                   # stable key — NOT masked
        F.lit(REDACT).alias("email"),                # PII → redact
        F.lit(REDACT).alias("full_name"),            # PII → redact
        F.expr(_mask_json("profile_json")).alias("profile_json"),  # in-JSON PII → redact
        "revenue", "event_ts", "_ingest_ts",         # non-PII → preserve
    )

## 2. Silver — AUTO CDC / SCD type 1 (the flow that was failing)

`silver_user` is the SCD1 dimension. It is fed by an **AUTO CDC flow** keyed on
`user_id` (the correct pattern — it absorbs updates/deletes that arrive as CDC
rows). This is the exact flow that was failing in the incident
(`dbo_user_silver_cdc`).

Two things make it robust:

1. The **streaming read of `bronze_user` uses `skipChangeCommits`** — so an
   erasure `DELETE` on bronze does not fatally fail this flow.
2. `dlt.apply_changes(... stored_as_scd_type=1)` keeps only the latest row per
   `user_id`.

In [ ]:
dlt.create_streaming_table(
    name="silver_user",
    comment="SCD type 1 dimension via AUTO CDC, keyed on user_id. Fed by a skipChangeCommits stream.",
    table_properties={"quality": "silver"},
)

@dlt.view(name="bronze_user_changes")
def bronze_user_changes():
    # The CDC feed for AUTO CDC. skipChangeCommits => a DELETE on bronze is skipped
    # here (not turned into a silver delete); the erasure notebook deletes silver
    # explicitly, so every layer is still erased. This is what keeps the stream alive.
    return (
        spark.readStream
        .option("skipChangeCommits", "true")   # ← survive erasure on bronze_user
        .table(f"{FQ}.bronze_user")
    )

dlt.apply_changes(
    target="silver_user",
    source="bronze_user_changes",
    keys=["user_id"],
    sequence_by=F.col("_ingest_ts"),
    stored_as_scd_type=1,
)

## 3. Gold — materialized view (NOT a streaming read)

`gold_user` is a per-customer lifetime rollup (lifetime revenue, event count). It
is a **materialized view** — a **batch** `spark.read` over `bronze_user`, not a
`spark.readStream`.

**Why an MV here instead of a third `skipChangeCommits` stream — this is the
precise answer to "do we need it on both layers?":**

- The two **append-only** hops (`raw → bronze`, `bronze → silver`) are genuine
  streams whose only non-append commits are erasures → `skipChangeCommits` is
  exactly right there.
- Gold aggregates over data that changes for **ordinary** reasons (SCD1 updates,
  and we want lifetime totals over *all* bronze events). A streaming read with
  `skipChangeCommits` would silently **drop legitimate updates**, and a streaming
  *aggregation* keeps checkpoint state that could **resurrect an erased subject**
  after a restart. A materialized view sidesteps both: it **fully recomputes**
  from the current bronze, so it reflects every erasure automatically and holds
  no state to resurrect. No `skipChangeCommits` needed.

So: **two streaming hops need `skipChangeCommits`; the aggregating gold layer
should be a materialized view.** We aggregate over **bronze** (all events) so
`lifetime_revenue` is a true lifetime sum, not a single SCD1 row.

In [ ]:
@dlt.table(
    name="gold_user",
    comment="Materialized view: per-customer lifetime rollup, recomputed from bronze each refresh.",
    table_properties={"quality": "gold"},
)
def gold_user():
    # batch read (spark.read, not readStream) => this is a materialized view.
    # Recomputes fully from bronze, so erasures propagate with no checkpoint state.
    return (
        spark.read.table(f"{FQ}.bronze_user")
        .groupBy("user_id")
        .agg(
            F.sum("revenue").alias("lifetime_revenue"),
            F.count("*").alias("event_count"),
            F.max("event_ts").alias("last_event_ts"),
        )
    )

## Notes for deployment

- **Serverless** Lakeflow (or Pro/Advanced) is required for AUTO CDC.
- After the first successful update, the pipeline runs **continuously or
  triggered**; the erasure in `02` runs against the published tables while this
  keeps streaming.
- If you ever *remove* `skipChangeCommits` and the pipeline has already met a
  non-append commit, you must **full-refresh the affected flow** to reset its
  checkpoint. With `skipChangeCommits` set from the start (this notebook), you
  never need that.